In [1]:
!pip install ultralytics onnx onnxruntime onnx-simplifier

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 121.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 110.8 MB/s eta 0:00:00
  Created wheel for onnx-simplifier: filename=onnx_simplifier-0.5.0-py3-none-any.whl size=1020 sha256=a8192aabda560526b85f5af7a3e7beb6dca8f881d02b98c8f485d873d936a999
  Stored in directory: /root/.cache/pip/wheels/f4/b8/30/e20e58d44e2c649a7bab0fed5acb4b883fc7bb2fa7fc9dde55
Successfully built onnx-simplifier


In [2]:
import torch
import torch.nn.utils.prune as prune
from ultralytics import YOLO
import shutil
import os
from onnxruntime.quantization import QuantFormat

import cv2
import numpy as np
from onnxruntime.quantization import (
    quantize_static,
    CalibrationDataReader,
    QuantType,
    QuantFormat,
    CalibrationMethod


)
from ultralytics.utils.downloads import download
from google.colab import files

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [3]:
import pandas as pd

splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/rishitdagli/cppe-5/" + splits["train"])

In [4]:
df.to_csv('data_cppe.csv')

In [5]:
!pip install -q datasets

import os
import yaml
from datasets import load_dataset

def download_and_convert_hf():
    dataset = load_dataset("rishitdagli/cppe-5")

    dataset_dir = '/content/cppe5_yolo'
    classes = ['Coverall', 'Face_Shield', 'Gloves', 'Goggles', 'Mask']

    for split in ['train', 'test']:
        yolo_split = 'val' if split == 'test' else 'train'
        os.makedirs(f"{dataset_dir}/images/{yolo_split}", exist_ok=True)
        os.makedirs(f"{dataset_dir}/labels/{yolo_split}", exist_ok=True)

        for i, item in enumerate(dataset[split]):
            img = item['image']

            if img.mode != 'RGB':
                img = img.convert('RGB')

            w, h = img.size

            img_path = f"{dataset_dir}/images/{yolo_split}/{i}.jpg"
            img.save(img_path)

            label_path = f"{dataset_dir}/labels/{yolo_split}/{i}.txt"
            with open(label_path, 'w') as f:
                for bbox, category in zip(item['objects']['bbox'], item['objects']['category']):
                    x_min, y_min, box_w, box_h = bbox

                    x_center = (x_min + box_w / 2) / w
                    y_center = (y_min + box_h / 2) / h
                    norm_w = box_w / w
                    norm_h = box_h / h

                    f.write(f"{category} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}\n")

    yaml_content = {
        'path': dataset_dir,
        'train': 'images/train',
        'val': 'images/val',
        'names': {i: name for i, name in enumerate(classes)}
    }

    config_path = os.path.join(dataset_dir, 'dataset.yaml')
    with open(config_path, 'w') as f:
        yaml.dump(yaml_content, f, sort_keys=False)

download_and_convert_hf()

README.md:   0%|          | 0.00/11.2k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  237MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.14MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/29 [00:00<?, ? examples/s]

In [6]:
!pip install -q datasets scikit-learn

import os
import yaml
from datasets import load_dataset
from sklearn.model_selection import train_test_split

def prepare_three_way_split():
    dataset = load_dataset("rishitdagli/cppe-5")

    dataset_dir = '/content/cppe5_yolo'
    classes = ['Coverall', 'Face_Shield', 'Gloves', 'Goggles', 'Mask']


    hf_train_full = list(dataset['train'])
    train_items, val_items = train_test_split(hf_train_full, test_size=0.1, random_state=42)


    test_items = list(dataset['test'])

    splits = {
        'train': train_items,
        'val': val_items,
        'test': test_items
    }

    for split_name, items in splits.items():
        os.makedirs(f"{dataset_dir}/images/{split_name}", exist_ok=True)
        os.makedirs(f"{dataset_dir}/labels/{split_name}", exist_ok=True)

        for i, item in enumerate(items):
            img = item['image']

            if img.mode != 'RGB':
                img = img.convert('RGB')

            w, h = img.size

            img_path = f"{dataset_dir}/images/{split_name}/{i}.jpg"
            img.save(img_path)

            label_path = f"{dataset_dir}/labels/{split_name}/{i}.txt"
            with open(label_path, 'w') as f:
                for bbox, category in zip(item['objects']['bbox'], item['objects']['category']):
                    x_min, y_min, box_w, box_h = bbox

                    x_center = (x_min + box_w / 2) / w
                    y_center = (y_min + box_h / 2) / h
                    norm_w = box_w / w
                    norm_h = box_h / h

                    f.write(f"{category} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}\n")

    yaml_content = {
        'path': dataset_dir,
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'names': {i: name for i, name in enumerate(classes)}
    }

    config_path = os.path.join(dataset_dir, 'dataset.yaml')
    with open(config_path, 'w') as f:
        yaml.dump(yaml_content, f, sort_keys=False)

prepare_three_way_split()

In [7]:
model = YOLO('yolo11n.pt')
results = model.train(
    data='/content/cppe5_yolo/dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=32,
    workers=8,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='edge_ai_project',
    name='cppe5_finetune'
)

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/cppe5_yolo/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=cppe5_finetune, nbs=64, nms=None, opset=None, optimize=F

In [ ]:
import os
import time
import cv2
import numpy as np
import shutil
import onnxruntime as ort
from ultralytics import YOLO
from onnxruntime.quantization import (
    quantize_static,
    CalibrationDataReader,
    QuantType,
    QuantFormat,
    CalibrationMethod
)

class YoloCalibrationDataReader(CalibrationDataReader):
    def __init__(self, image_folder, input_shape=(1, 3, 640, 640)):
        self.image_files = [os.path.join(image_folder, f)
                            for f in os.listdir(image_folder) if f.endswith(('.jpg', '.jpeg'))][:50]
        self.input_shape = input_shape
        self.enum_data_dicts = iter(self.read_images())

    def get_next(self):
        return next(self.enum_data_dicts, None)

    def read_images(self):
        for image_path in self.image_files:
            img = cv2.imread(image_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (self.input_shape[3], self.input_shape[2]))
            img = img.transpose((2, 0, 1))
            yield {"images": np.expand_dims(img, axis=0).astype(np.float32) / 255.0}

best_weights = '/content/runs/detect/edge_ai_project/cppe5_finetune/weights/best.pt'
dataset_yaml = '/content/cppe5_yolo/dataset.yaml'
calib_folder = '/content/cppe5_yolo/images/train'

model = YOLO(best_weights)
export_path = model.export(format='onnx', opset=13, batch=1, simplify=True, dynamic=False)

fp32_model = '/content/cppe5_fp32.onnx'
shutil.copy(export_path, fp32_model)

methods = {
    "MinMax": CalibrationMethod.MinMax,
    "Entropy": CalibrationMethod.Entropy,
    "Percentile": CalibrationMethod.Percentile
}

quantized_models = {}

for name, method in methods.items():
    output_path = f'/content/cppe5_int8_{name.lower()}.onnx'
    dr = YoloCalibrationDataReader(calib_folder)

    quantize_static(
        model_input=fp32_model,
        model_output=output_path,
        calibration_data_reader=dr,
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QUInt8,
        weight_type=QuantType.QInt8,
        per_channel=True,
        reduce_range=True,
        # Разрешаем сжатие исключительно для сверток и линейных слоев
        op_types_to_quantize=['Conv', 'MatMul'],
        calibrate_method=method
    )
    quantized_models[name] = output_path

print("\n" + "="*60)
print("Banchmark rez (LATENCY&ACCURACY TEST)")
print("="*60)

dummy_input = np.random.randn(1, 3, 640, 640).astype(np.float32)
models_to_test = {"FP32_Baseline": fp32_model, **quantized_models}

for model_name, path in models_to_test.items():
    session = ort.InferenceSession(path, providers=['CPUExecutionProvider'])
    input_name = session.get_inputs()[0].name
    for _ in range(5): session.run(None, {input_name: dummy_input})

    start_time = time.time()
    for _ in range(50): session.run(None, {input_name: dummy_input})
    avg_latency = (time.time() - start_time) / 50 * 1000

    val_model = YOLO(path, task='detect')
    metrics = val_model.val(data=dataset_yaml, imgsz=640, split='test', plots=False, verbose=False)

    print(f"{model_name:15} | Latency: {avg_latency:5.1f} мс | mAP50: {metrics.box.map50:.4f}")

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n summary (fused): 100 layers, 2,583,127 parameters, 0 gradients, 6.4 GFLOPs

PyTorch: starting from '/content/runs/detect/edge_ai_project/cppe5_finetune/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 9, 8400) (5.2 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 10 packages in 391ms
Prepared 2 packages in 34ms
Installed 2 packages in 10ms
 + colorama==0.4.6
 + onnxslim==0.1.98

requirements: AutoUpdate success ✅ 1.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.23.2 opset 13...
ONNX: slimming with onnxslim 0.1.98...
ONNX: export success ✅ 7.5s, save

In [ ]:
from google.colab import files
files.download('/content/cppe5_int8_minmax.onnx')